# D1 · QGIS native algorithms — buffers, overlays, repair, spatial joins

**What QGIS native algorithms are:** the tools of QGIS's Processing Toolbox with ids starting `native:` (written in C++, fast)
or `qgis:`. They cover almost everything you did with GeoPandas: buffer, clip, intersection, dissolve, spatial joins, field calculator…
plus things GeoPandas does not have built in, such as **network service areas** and **clustering**.

**Daily picture:** GeoPandas is cooking from a recipe you write yourself. QGIS native is a set of kitchen machines with buttons.
Same food, different way of working. Knowing both means you can read and repeat a colleague's QGIS workflow in Python.

**How we call them:** `qrun("native:buffer", INPUT=..., DISTANCE=..., OUTPUT=...)` returns the output paths.
`qhelp("native:buffer")` shows the parameters, the same fields you see in the QGIS dialog.
A layer inside a GeoPackage is written `file.gpkg|layername=schools`; the helper `V("schools")` does that for you.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
from geotrain.desktop import qrun, qhelp, qlist, sh, grass_alg, enable_grass, qgis_process_path, quote, minimal_project
OUT = DATA_DIR / "d_out"
OUT.mkdir(exist_ok=True)
V = lambda layer: f"{GPKG}|layername={layer}"       # how QGIS names a layer inside a GeoPackage
DEM = DATA_DIR / "dem.tif"
print("qgis_process:", qgis_process_path())

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
nbh = gpd.read_file(GPKG, layer="neighbourhoods")

---
## Level 1 — Basics: find, read and run a tool

**Goal of this level:** find an algorithm, read its parameters, run it and open the result.

**Daily-life picture:** Finding the right machine in a workshop, reading its instruction plate, and pressing start.

### Exercise 1.1 — Find tools by keyword

**Step 1 · Purpose.** What is the **purpose** of `qlist('buffer')`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`qlist("buffer")` returns `[id, name]` pairs. Notice buffers exist in several providers (native, gdal, grass).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
for tool_id, name in qlist("____"):
    print(tool_id, "—", name)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Searches the Processing Toolbox by id or name, like the search box in QGIS.

**Real life:** You know *what* you want (a buffer) but not the tool's exact name.

```python
for tool_id, name in qlist("buffer"):
    print(tool_id, "—", name)
```

</details>

### Exercise 1.2 — Read a tool's parameters

**Step 1 · Purpose.** What is the **purpose** of `qhelp('native:buffer')`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`qhelp("native:buffer")`. Find DISTANCE, SEGMENTS, DISSOLVE and OUTPUT.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
qhelp("____")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Shows each parameter's name, meaning, type and allowed values (the text form of the QGIS dialog).

**Real life:** Before automating a tool, you check which parameters are required and what the codes mean (e.g. END_CAP_STYLE 0 = round).

```python
qhelp("native:buffer")
```

</details>

### Exercise 1.3 — Run a buffer and open the result

**Step 1 · Purpose.** What is the **purpose** of `qrun('native:buffer', ...)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`INPUT=V("schools")`, `DISTANCE=600`, `SEGMENTS=16`, `DISSOLVE=False`, `OUTPUT=str(OUT / "school_buf.gpkg")`. Then `gpd.read_file(res["OUTPUT"])`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:buffer", INPUT=V("schools"), DISTANCE=____, SEGMENTS=16, DISSOLVE=False,
           OUTPUT=str(OUT / "school_buf.gpkg"))
buf = gpd.read_file(res["____"])
ax = nbh.boundary.plot(color="grey"); buf.plot(ax=ax, alpha=.4); plt.show()
buf[["school"]].assign(area=buf.area.round())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Runs the algorithm and writes the output file; `qrun` returns a dict of output paths.

**Real life:** The 600 m catchment of each school, made by QGIS but scripted, so you can re-run it when the schools change.

```python
res = qrun("native:buffer", INPUT=V("schools"), DISTANCE=600, SEGMENTS=16, DISSOLVE=False,
           OUTPUT=str(OUT / "school_buf.gpkg"))
buf = gpd.read_file(res["OUTPUT"])
ax = nbh.boundary.plot(color="grey"); buf.plot(ax=ax, alpha=.4); plt.show()
buf[["school"]].assign(area=buf.area.round())
```

Compare with GeoPandas A3 2.3 (`schools.buffer(600)`): same shapes, because both use GEOS.

</details>

---
## Level 2 — Core tools: repair, select, calculate, dissolve

**Goal of this level:** clean geometries and build new fields and layers with native tools.

**Daily-life picture:** Tidying a messy spreadsheet: fix broken rows, filter, add a formula column, then group.

### Exercise 2.1 — Fix broken geometries

**Step 1 · Purpose.** What is the **purpose** of `native:fixgeometries`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

First we write a layer with one bow-tie using GeoPandas. Then run `native:fixgeometries` with `INPUT` and `OUTPUT`. Check `.is_valid` before and after.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
from shapely import Polygon
bad = gpd.GeoDataFrame({"id": [1, 2]}, geometry=[Polygon([(0, 0), (10, 10), (10, 0), (0, 10)]),
                                                 Polygon([(20, 0), (30, 0), (30, 10), (20, 10)])], crs=32633)
bad.to_file(OUT / "bad.gpkg", layer="bad")
res = qrun("native:____", INPUT=str(OUT / "bad.gpkg"), OUTPUT=str(OUT / "fixed.gpkg"))
fixed = gpd.read_file(res["OUTPUT"])
print(bad.is_valid.tolist(), "->", fixed.is_valid.tolist(), fixed.area.tolist())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Repairs invalid geometries (self-crossings, bow-ties) so later tools do not fail or give wrong areas.

**Real life:** Hand-digitised parcels from a city office often contain bow-ties; fix them before any overlay.

```python
from shapely import Polygon
bad = gpd.GeoDataFrame({"id": [1, 2]}, geometry=[Polygon([(0, 0), (10, 10), (10, 0), (0, 10)]),
                                                 Polygon([(20, 0), (30, 0), (30, 10), (20, 10)])], crs=32633)
bad.to_file(OUT / "bad.gpkg", layer="bad")
res = qrun("native:fixgeometries", INPUT=str(OUT / "bad.gpkg"), OUTPUT=str(OUT / "fixed.gpkg"))
fixed = gpd.read_file(res["OUTPUT"])
print(bad.is_valid.tolist(), "->", fixed.is_valid.tolist(), fixed.area.tolist())
```

Twins: Shapely `make_valid` (A1 3.7), PostGIS `ST_MakeValid` (B1 1.4).

</details>

### Exercise 2.2 — Select by expression

**Step 1 · Purpose.** What is the **purpose** of `native:extractbyexpression`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

QGIS expressions put **field names in double quotes** and **text in single quotes**: `EXPRESSION='"road_type" = \'primary\''`. In Python, write the whole expression inside a pair of `"""..."""` or use single quotes outside.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:extractbyexpression", INPUT=V("roads"),
           EXPRESSION="""____""", OUTPUT=str(OUT / "primary.gpkg"))
gpd.read_file(res["OUTPUT"])[["name", "road_type"]]

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Keeps only features that match a QGIS expression (the same language as QGIS's 'Select by expression').

**Real life:** Only the primary roads, only the accidents at night, only parcels over 1 ha.

```python
res = qrun("native:extractbyexpression", INPUT=V("roads"),
           EXPRESSION=""""road_type" = 'primary'""", OUTPUT=str(OUT / "primary.gpkg"))
gpd.read_file(res["OUTPUT"])[["name", "road_type"]]
```

</details>

### Exercise 2.3 — Field calculator

**Step 1 · Purpose.** What is the **purpose** of `native:fieldcalculator`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`FIELD_NAME="density"`, `FIELD_TYPE=0` (0 = decimal number), `FORMULA='"population" / ($area / 1e6)'`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:fieldcalculator", INPUT=V("neighbourhoods"), FIELD_NAME="density", FIELD_TYPE=0,
           FIELD_LENGTH=10, FIELD_PRECISION=1, FORMULA=""""population" / (____ / 1e6)""",
           OUTPUT=str(OUT / "nbh_density.gpkg"))
gpd.read_file(res["OUTPUT"])[["name", "population", "density"]]

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Adds a new field computed with a QGIS expression; `$area`, `$length`, `$x` give geometry values.

**Real life:** Population density per neighbourhood; a label field; a class (e.g. 'north/centre/south').

```python
res = qrun("native:fieldcalculator", INPUT=V("neighbourhoods"), FIELD_NAME="density", FIELD_TYPE=0,
           FIELD_LENGTH=10, FIELD_PRECISION=1, FORMULA=""""population" / ($area / 1e6)""",
           OUTPUT=str(OUT / "nbh_density.gpkg"))
gpd.read_file(res["OUTPUT"])[["name", "population", "density"]]
```

</details>

### Exercise 2.4 — Chain two tools: classify, then dissolve

**Step 1 · Purpose.** What is the **purpose** of `fieldcalculator → native:dissolve`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Step 1: text field `district` (`FIELD_TYPE=2` = text) with `CASE WHEN "nb_id" <= 3 THEN 'south' WHEN "nb_id" <= 6 THEN 'centre' ELSE 'north' END`. Step 2: `native:dissolve` with `FIELD="district"`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
step1 = qrun("native:fieldcalculator", INPUT=V("neighbourhoods"), FIELD_NAME="district", FIELD_TYPE=2,
             FIELD_LENGTH=10, FIELD_PRECISION=0,
             FORMULA="""CASE WHEN "nb_id" <= 3 THEN 'south' WHEN "nb_id" <= 6 THEN 'centre' ELSE 'north' END""",
             OUTPUT=str(OUT / "nbh_district.gpkg"))
step2 = qrun("native:____", INPUT=step1["OUTPUT"], FIELD="____", OUTPUT=str(OUT / "districts.gpkg"))
d = gpd.read_file(step2["OUTPUT"])
d.plot(column="district", legend=True, edgecolor="black"); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** The output of one tool is the input of the next — a small pipeline (what the QGIS Graphical Modeler builds with boxes and arrows).

**Real life:** Build 3 districts from 9 neighbourhoods, in a way you can re-run next year.

```python
step1 = qrun("native:fieldcalculator", INPUT=V("neighbourhoods"), FIELD_NAME="district", FIELD_TYPE=2,
             FIELD_LENGTH=10, FIELD_PRECISION=0,
             FORMULA="""CASE WHEN "nb_id" <= 3 THEN 'south' WHEN "nb_id" <= 6 THEN 'centre' ELSE 'north' END""",
             OUTPUT=str(OUT / "nbh_district.gpkg"))
step2 = qrun("native:dissolve", INPUT=step1["OUTPUT"], FIELD="district", OUTPUT=str(OUT / "districts.gpkg"))
d = gpd.read_file(step2["OUTPUT"])
d.plot(column="district", legend=True, edgecolor="black"); plt.show()
```

QGIS's dissolve keeps the attributes of the first feature; it does not sum population (GeoPandas `dissolve(aggfunc=...)` can). For sums use *Aggregate* (`native:aggregate`).

</details>

---
## Level 3 — Combining: spatial joins, overlays, raster statistics

**Goal of this level:** answer the A3 questions with QGIS tools, and add raster values to vectors.

**Daily-life picture:** Same questions, a different toolbox — so you can check one tool's answer with the other.

### Exercise 3.1 — Join attributes by location

**Step 1 · Purpose.** What is the **purpose** of `native:joinattributesbylocation`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`INPUT=V("accidents")`, `JOIN=V("neighbourhoods")`, `PREDICATE=5` (5 = 'are within'; 0 = intersect), `JOIN_FIELDS="name"`, `METHOD=0` (one-to-many).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:joinattributesbylocation", INPUT=V("accidents"), JOIN=V("neighbourhoods"),
           PREDICATE=____, JOIN_FIELDS="name", METHOD=0, DISCARD_NONMATCHING=False,
           OUTPUT=str(OUT / "acc_nbh.gpkg"))
gpd.read_file(res["OUTPUT"])["name"].value_counts()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Adds the attributes of one layer to another where their geometries meet a spatial rule (a spatial join, like `gpd.sjoin`).

**Real life:** Give every accident the name of its neighbourhood.

```python
res = qrun("native:joinattributesbylocation", INPUT=V("accidents"), JOIN=V("neighbourhoods"),
           PREDICATE=5, JOIN_FIELDS="name", METHOD=0, DISCARD_NONMATCHING=False,
           OUTPUT=str(OUT / "acc_nbh.gpkg"))
gpd.read_file(res["OUTPUT"])["name"].value_counts()
```

</details>

### Exercise 3.2 — Count points in polygons

**Step 1 · Purpose.** What is the **purpose** of `native:countpointsinpolygon`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`POLYGONS=V("neighbourhoods")`, `POINTS=V("accidents")`, `FIELD="n_acc"`. You can also give `WEIGHT` (sum a field instead of counting).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:countpointsinpolygon", POLYGONS=V("neighbourhoods"), POINTS=V("____"), FIELD="n_acc",
           OUTPUT=str(OUT / "nbh_counts.gpkg"))
cnt = gpd.read_file(res["OUTPUT"])
cnt[["name", "n_acc"]].sort_values("n_acc", ascending=False)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Counts how many points fall in each polygon and writes the count as a new field (join + groupby in one step).

**Real life:** Shops per district, trees per park, accidents per neighbourhood.

```python
res = qrun("native:countpointsinpolygon", POLYGONS=V("neighbourhoods"), POINTS=V("accidents"), FIELD="n_acc",
           OUTPUT=str(OUT / "nbh_counts.gpkg"))
cnt = gpd.read_file(res["OUTPUT"])
cnt[["name", "n_acc"]].sort_values("n_acc", ascending=False)
```

</details>

### Exercise 3.3 — Overlay: which roads flood?

**Step 1 · Purpose.** What is the **purpose** of `native:buffer → native:clip`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Buffer the river by 150 m first, then `native:clip` with `INPUT=V("roads")` and `OVERLAY=` the buffer output.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
flood = qrun("native:buffer", INPUT=V("river"), DISTANCE=150, SEGMENTS=16, DISSOLVE=True, OUTPUT=str(OUT / "flood.gpkg"))
res = qrun("native:____", INPUT=V("roads"), OVERLAY=flood["OUTPUT"], OUTPUT=str(OUT / "wet_roads.gpkg"))
wet = gpd.read_file(res["OUTPUT"])
wet.assign(km=wet.length / 1000).groupby("road_type")["km"].sum().round(2)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Clip keeps only the parts of a layer inside a mask polygon.

**Real life:** Road length under water if the river floods (A3 3.4, now with QGIS tools).

```python
flood = qrun("native:buffer", INPUT=V("river"), DISTANCE=150, SEGMENTS=16, DISSOLVE=True, OUTPUT=str(OUT / "flood.gpkg"))
res = qrun("native:clip", INPUT=V("roads"), OVERLAY=flood["OUTPUT"], OUTPUT=str(OUT / "wet_roads.gpkg"))
wet = gpd.read_file(res["OUTPUT"])
wet.assign(km=wet.length / 1000).groupby("road_type")["km"].sum().round(2)
```

</details>

### Exercise 3.4 — Nearest feature join

**Step 1 · Purpose.** What is the **purpose** of `native:joinbynearest`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`INPUT=V("schools")`, `INPUT_2=V("clinics")`, `FIELDS_TO_COPY="clinic"`, `NEIGHBORS=1`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:joinbynearest", INPUT=V("schools"), INPUT_2=V("clinics"), FIELDS_TO_COPY="clinic",
           NEIGHBORS=____, DISCARD_NONMATCHING=False, OUTPUT=str(OUT / "school_clinic.gpkg"))
gpd.read_file(res["OUTPUT"])[["school", "clinic", "distance"]].round()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Joins each feature to its nearest feature(s) in another layer and adds a `distance` field (like `gpd.sjoin_nearest`).

**Real life:** Nearest clinic for each school, with the distance.

```python
res = qrun("native:joinbynearest", INPUT=V("schools"), INPUT_2=V("clinics"), FIELDS_TO_COPY="clinic",
           NEIGHBORS=1, DISCARD_NONMATCHING=False, OUTPUT=str(OUT / "school_clinic.gpkg"))
gpd.read_file(res["OUTPUT"])[["school", "clinic", "distance"]].round()
```

</details>

### Exercise 3.5 — Zonal statistics (raster values per polygon)

**Step 1 · Purpose.** What is the **purpose** of `native:zonalstatisticsfb`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`INPUT=V("neighbourhoods")`, `INPUT_RASTER=str(DEM)`, `RASTER_BAND=1`, `COLUMN_PREFIX="elev_"`, `STATISTICS=[2, 6]` (2 = mean, 6 = max).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
res = qrun("native:zonalstatisticsfb", INPUT=V("neighbourhoods"), INPUT_RASTER=str(DEM), RASTER_BAND=1,
           COLUMN_PREFIX="elev_", STATISTICS=[____, 6], OUTPUT=str(OUT / "nbh_elev.gpkg"))
gpd.read_file(res["OUTPUT"])[["name", "elev_mean", "elev_max"]].round(1)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes statistics of raster cells inside each polygon (mean, max…) and writes them as new fields.

**Real life:** Mean elevation of each neighbourhood (A5 3.4 in one tool).

```python
res = qrun("native:zonalstatisticsfb", INPUT=V("neighbourhoods"), INPUT_RASTER=str(DEM), RASTER_BAND=1,
           COLUMN_PREFIX="elev_", STATISTICS=[2, 6], OUTPUT=str(OUT / "nbh_elev.gpkg"))
gpd.read_file(res["OUTPUT"])[["name", "elev_mean", "elev_max"]].round(1)
```

</details>

---
## Level 4 — Professional: networks, clusters, reproducible pipelines

**Goal of this level:** use QGIS tools that go beyond GeoPandas, and build a pipeline you can check against Python.

**Daily-life picture:** Using a GPS route planner instead of a ruler: the real path, not the straight line.

### Exercise 4.1 — Walking along streets vs. a circle

**Scenario.** From each clinic, which roads can you reach within **800 m along the road network**? Compare the network result with the simple 800 m circle:
how many km of road are inside the circle, and how many are really reachable?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`native:serviceareafromlayer`: `INPUT=V("roads")` (the network), `START_POINTS=V("clinics")`, `STRATEGY=0` (shortest = distance),
`TRAVEL_COST2=800`, `OUTPUT_LINES=...`. Network tools need a QGIS project: pass `_project=minimal_project()`. For the circle: buffer clinics 800 m and clip the roads.
Daily picture: a circle on a map says 'close', but a river without a bridge can make a place 3 km away on foot.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
net = qrun("native:serviceareafromlayer", _project=minimal_project(), INPUT=V("roads"), STRATEGY=0, START_POINTS=V("clinics"),
           TRAVEL_COST2=____, TOLERANCE=1, DEFAULT_DIRECTION=2, DEFAULT_SPEED=5,
           OUTPUT_LINES=str(OUT / "service_lines.gpkg"))
circ = qrun("native:buffer", INPUT=V("clinics"), DISTANCE=800, SEGMENTS=16, DISSOLVE=True, OUTPUT=str(OUT / "clinic800.gpkg"))
inside = qrun("native:clip", INPUT=V("roads"), OVERLAY=circ["OUTPUT"], OUTPUT=str(OUT / "roads_in_circle.gpkg"))
lines = gpd.read_file(net["OUTPUT_LINES"]); roads_c = gpd.read_file(inside["OUTPUT"])
print(f"road inside circles: {roads_c.length.sum() / 1000:.1f} km | reachable on the network: {lines.dissolve().length.sum() / 1000:.1f} km")
ax = gpd.read_file(circ["OUTPUT"]).plot(alpha=.2, figsize=(6, 6)); gpd.read_file(GPKG, layer="roads").plot(ax=ax, color="lightgrey")
lines.plot(ax=ax, color="red", linewidth=2); gpd.read_file(GPKG, layer="clinics").plot(ax=ax, color="black"); plt.show()

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Proximity (network vs. buffer model)

The network reaches less than the circle suggests: clinics that are not on a road first need a 'connector', and the network only follows real streets.
This is the network model from the A1 test (Model 2), done with a ready-made QGIS tool.
Limitation: Riverton's road layer has only 7 main roads; with a full street network (e.g. from OpenStreetMap) the result is much more realistic.

```python
net = qrun("native:serviceareafromlayer", _project=minimal_project(), INPUT=V("roads"), STRATEGY=0, START_POINTS=V("clinics"),
           TRAVEL_COST2=800, TOLERANCE=1, DEFAULT_DIRECTION=2, DEFAULT_SPEED=5,
           OUTPUT_LINES=str(OUT / "service_lines.gpkg"))
circ = qrun("native:buffer", INPUT=V("clinics"), DISTANCE=800, SEGMENTS=16, DISSOLVE=True, OUTPUT=str(OUT / "clinic800.gpkg"))
inside = qrun("native:clip", INPUT=V("roads"), OVERLAY=circ["OUTPUT"], OUTPUT=str(OUT / "roads_in_circle.gpkg"))
lines = gpd.read_file(net["OUTPUT_LINES"]); roads_c = gpd.read_file(inside["OUTPUT"])
print(f"road inside circles: {roads_c.length.sum() / 1000:.1f} km | reachable on the network: {lines.dissolve().length.sum() / 1000:.1f} km")
ax = gpd.read_file(circ["OUTPUT"]).plot(alpha=.2, figsize=(6, 6)); gpd.read_file(GPKG, layer="roads").plot(ax=ax, color="lightgrey")
lines.plot(ax=ax, color="red", linewidth=2); gpd.read_file(GPKG, layer="clinics").plot(ax=ax, color="black"); plt.show()
```

**Why this matters:** Service areas / isochrones are the standard accessibility model in planning (15-minute city, emergency response).

</details>

### Exercise 4.2 — Where do accidents cluster? (DBSCAN)

**Scenario.** Find groups of accidents where at least **8** accidents lie within **150 m** of each other. How many clusters are there, and where?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`native:dbscanclustering`: `EPS=150` (the 150 m), `MIN_SIZE=8`, `FIELD_NAME="cluster"`. Points that belong to no cluster get an empty value (noise).

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
res = qrun("native:dbscanclustering", INPUT=V("accidents"), MIN_SIZE=____, EPS=____, FIELD_NAME="cluster",
           SIZE_FIELD_NAME="size", OUTPUT=str(OUT / "acc_clusters.gpkg"))
cl = gpd.read_file(res["OUTPUT"])
print(cl["cluster"].nunique(), "clusters;", cl["cluster"].isna().sum(), "accidents are noise")
ax = nbh.boundary.plot(color="grey", figsize=(6, 6)); gpd.read_file(GPKG, layer="roads").plot(ax=ax, color="lightgrey")
cl[cl.cluster.isna()].plot(ax=ax, color="black", markersize=3); cl.dropna(subset=["cluster"]).plot(ax=ax, column="cluster", categorical=True, markersize=12); plt.show()

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Descriptive / statistical (density-based clustering)

DBSCAN finds dense groups (here the dangerous junctions) and labels scattered points as noise.
It *describes* clusters but does not test them against chance — for that use PySAL (A6: Gi*, LISA, Ripley's K). The result also depends on EPS and MIN_SIZE: try 100 m and 250 m.

```python
res = qrun("native:dbscanclustering", INPUT=V("accidents"), MIN_SIZE=8, EPS=150, FIELD_NAME="cluster",
           SIZE_FIELD_NAME="size", OUTPUT=str(OUT / "acc_clusters.gpkg"))
cl = gpd.read_file(res["OUTPUT"])
print(cl["cluster"].nunique(), "clusters;", cl["cluster"].isna().sum(), "accidents are noise")
ax = nbh.boundary.plot(color="grey", figsize=(6, 6)); gpd.read_file(GPKG, layer="roads").plot(ax=ax, color="lightgrey")
cl[cl.cluster.isna()].plot(ax=ax, color="black", markersize=3); cl.dropna(subset=["cluster"]).plot(ax=ax, column="cluster", categorical=True, markersize=12); plt.show()
```

</details>

### Exercise 4.3 — Same question, two toolboxes: does QGIS agree with GeoPandas?

**Scenario.** Count accidents per neighbourhood with QGIS (3.2) and with GeoPandas `sjoin`. Write a check that fails if any count differs.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Merge both results on `name` and compare with `assert (a == b).all()`. Daily picture: weighing a parcel on two scales before posting it.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
acc = gpd.read_file(GPKG, layer="accidents")
gp = gpd.sjoin(acc, nbh[["name", "geometry"]], predicate="within")["name"].value_counts()
both = cnt[["name", "n_acc"]].assign(geopandas=cnt["name"].map(gp).fillna(0))
assert (both["n_acc"] == both["____"]).all(), "QGIS and GeoPandas disagree!"
print("✅ QGIS and GeoPandas agree"); both

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Reproducibility (method check)

Cross-checking two independent tools is a cheap and strong quality test. Small differences usually come from boundary rules (a point exactly on an edge) or buffer segments.

```python
acc = gpd.read_file(GPKG, layer="accidents")
gp = gpd.sjoin(acc, nbh[["name", "geometry"]], predicate="within")["name"].value_counts()
both = cnt[["name", "n_acc"]].assign(geopandas=cnt["name"].map(gp).fillna(0))
assert (both["n_acc"] == both["geopandas"]).all(), "QGIS and GeoPandas disagree!"
print("✅ QGIS and GeoPandas agree"); both
```

</details>

### The same tool in PyQGIS (for when you work inside QGIS)

In the QGIS Python console (or a Python that has `qgis` installed) the exact same call is:
```python
import processing
result = processing.run("native:buffer", {"INPUT": "riverton.gpkg|layername=schools", "DISTANCE": 600,
                                          "SEGMENTS": 16, "DISSOLVE": False, "OUTPUT": "memory:"})
layer = result["OUTPUT"]
```
Same ids, same parameter names. What you learn with `qrun()` transfers one-to-one.

---
## 🏁 Final test

Use QGIS native tools through `qrun()` (and GeoPandas only to look at results). For each: **question type → tool chain → code → interpretation.**

### Test 1 — 🛠️ Task question

**A.** Make multi-ring buffers (200, 400, 600 m) around the clinics and count the houses in each ring.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Type: proximity. Chain: multi-ring buffer → count points (weighted by residents).
```python
rings = qrun("native:multiringconstantbuffer", INPUT=V("clinics"), RINGS=3, DISTANCE=200, OUTPUT=str(OUT / "rings.gpkg"))
c = qrun("native:countpointsinpolygon", POLYGONS=rings["OUTPUT"], POINTS=V("houses"), WEIGHT="residents",
         FIELD="residents", OUTPUT=str(OUT / "rings_count.gpkg"))
print(gpd.read_file(c["OUTPUT"])[["clinic", "ringId", "residents"]])
```

</details>

### Test 2 — 🛠️ Task question

**B.** List the parks with the share of their area inside the 150 m flood zone, using QGIS tools only for the geometry work.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Type: overlay + measurement.
```python
inter = qrun("native:intersection", INPUT=V("parks"), OVERLAY=str(OUT / "flood.gpkg"), OUTPUT=str(OUT / "park_flood.gpkg"))
pf = gpd.read_file(inter["OUTPUT"]); parks = gpd.read_file(GPKG, layer="parks")
print((pf.dissolve("park").area / parks.set_index("park").area).fillna(0).round(3))
```

</details>

### Test 3 — 🛠️ Task question

**C.** Give every house the elevation of the DEM cell it stands on, and report the lowest 5 houses.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
s = qrun("native:rastersampling", INPUT=V("houses"), RASTERCOPY=str(DEM), COLUMN_PREFIX="elev_", OUTPUT=str(OUT / "houses_elev.gpkg"))
print(gpd.read_file(s["OUTPUT"]).nsmallest(5, "elev_1")[["house_id", "elev_1"]])
```

</details>

### Test 4 — 🗺️ Decision question

**D.** A colleague made a site-selection map by clicking through 12 QGIS tools. The council asks for the same map next year with new data. What do you recommend?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Turn the clicks into a script (`qrun` chain or PyQGIS) or a **Graphical Modeler** model (.model3), store it with the data, and re-run it.
Recommendation: a script in a notebook, because it documents every parameter, can be checked by others, and runs in one go.

</details>

### Test 5 — 🧠 Modelling question

**E · Modelling.** Riverton wants a map of *'how long does it take to walk to the nearest clinic'* for every house. Which QGIS tools would you chain, and what data would you need first?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Data: a complete **street network** (OpenStreetMap footways + streets), clinics, houses.
Chain: snap houses to the network → `native:shortestpathlayertopoint` or service areas with several costs (5, 10, 15 min at ~5 km/h) → join the band to each house (`joinattributesbylocation`) → summarise per neighbourhood.
Assumptions: walking speed, no slopes (GRASS `r.walk` can add slope effects, D3), barriers like rivers only crossable at bridges.

</details>

---
## 🏗️ Project — Fire-engine response times on the street network

**The brief.** Riverton has one fire station. The fire brigade's target: reach every building within **4 minutes** of driving.
        Using the new **street network** (with speeds and only 3 bridges over the river), compute the 2- and 4-minute service areas
        with QGIS's network tools, find the buildings outside the 4-minute area, and test three candidate sites for a second station.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `streets.gpkg` | 1 172 segments: `speed_kmh` (50 main / 30 local), `bridge` |
| `fire_stations.gpkg` | the current station |
| `buildings.gpkg` | ~1 700 buildings with residents |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()
P = lambda name, layer=None: f"{PROJ_DIR / name}" + (f"|layername={layer}" if layer else "")

import pandas as pd
streets = gpd.read_file(PROJ_DIR / "streets.gpkg")
bld = gpd.read_file(PROJ_DIR / "buildings.gpkg")
river = gpd.read_file(GPKG, layer="river").geometry.iloc[0]

### Project task 1

**Service areas.** With `native:serviceareafromlayer`, *fastest* path (`STRATEGY=1`), speed from `speed_kmh`: which streets are reachable in **2** and in **4 minutes**? (Travel cost for 'fastest' is in **hours**: 4 min = 4/60.) Report km of street reached (merge the output lines first).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
def service(minutes, start, name):
    r = qrun("native:serviceareafromlayer", _project=minimal_project(), INPUT=P("streets.gpkg"), STRATEGY=1,
             SPEED_FIELD="speed_kmh", DEFAULT_SPEED=30, DEFAULT_DIRECTION=2, TOLERANCE=1,
             START_POINTS=start, TRAVEL_COST2=minutes / 60, OUTPUT_LINES=str(OUT / f"{name}.gpkg"))
    return gpd.read_file(r["OUTPUT_LINES"]).dissolve()
sa2, sa4 = service(2, P("fire_stations.gpkg"), "fire_2min"), service(4, P("fire_stations.gpkg"), "fire_4min")
print("km reached: 2 min", round(sa2.length.sum() / 1000, 1), "| 4 min", round(sa4.length.sum() / 1000, 1),
      "| whole network", round(streets.length.sum() / 1000, 1))
```

</details>

### Project task 2

**Covered buildings.** A building is covered if it lies within **50 m** of a reachable street. Share of residents covered in 4 minutes?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
bld["covered4"] = bld.distance(sa4.geometry.iloc[0]) <= 50
res = bld[bld.residents > 0]
print(f"{res.loc[res.covered4, 'residents'].sum() / res.residents.sum():.1%} of residents covered in 4 min")
```

</details>

### Project task 3

**The river effect.** The station is south of the river. What **share** of residents is uncovered north vs south of the river? (Build a polygon 'north of the river line' and test `within`.) Compare shares, not counts — the two sides have different populations.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
import numpy as np
from shapely.geometry import Polygon
xs = np.array(river.coords)[:, 0]; ys = np.array(river.coords)[:, 1]
north_poly = Polygon(list(zip(xs, ys)) + [(xs[-1], 5_830_000), (xs[0], 5_830_000)])
side = np.where(res.within(north_poly), "north", "south")
out = res.assign(side=side, unc=~res.covered4).groupby("side").apply(
    lambda g: pd.Series({"residents": g.residents.sum(), "uncovered_share": round((g.residents * g.unc).sum() / g.residents.sum(), 2)}), include_groups=False)
out
```
The far (north) side has the larger uncovered share: with only 3 bridges, engines must detour. That is why a second station north of the river is worth testing next.

</details>

### Project task 4

**Where should station 2 go?** Test three candidates — (391 000, 5 823 000), (394 750, 5 822 500), (393 000, 5 819 000). For each, compute its 4-minute area and the residents **newly** covered (not covered by station 1). Pick the best.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
cands = gpd.GeoDataFrame({"cand": ["NW", "NE", "S"]},
                         geometry=gpd.points_from_xy([391_000, 394_750, 393_000], [5_823_000, 5_822_500, 5_819_000]), crs=32633)
gain = {}
for _, c in cands.iterrows():
    f = OUT / f"cand_{c.cand}.gpkg"
    cands[cands.cand == c.cand].to_file(f, layer="c")
    sa = service(4, str(f), f"cand_{c.cand}_4min")
    newly = res[(~res.covered4) & (res.distance(sa.geometry.iloc[0]) <= 50)]
    gain[c.cand] = int(newly.residents.sum())
print(gain, "-> best:", max(gain, key=gain.get))
```

</details>

### Project task 5

**Publish** the 4-minute area, covered/uncovered buildings and the candidates to PostGIS.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
publish({"fire_area_4min": sa4, "fire_buildings": bld[["bldg_id", "residents", "covered4", "geometry"]], "fire_candidates": cands})
```

</details>

### 🗺️ Project in QGIS

1. Load `fire_area_4min`, `fire_buildings` (style: `covered4` true = grey, false = red) and `fire_candidates` from **geotrain → projects**, plus `streets.gpkg` (bridges in thick blue: `"bridge" = true`).
2. In QGIS itself: *Processing Toolbox → Network analysis → Service area (from point)* — click on a candidate on the map as start point, choose `streets`, *Fastest*, speed field `speed_kmh`, travel cost `0.0667` (4 min). Same result as your script?
3. Build a **Graphical Model** (*Processing → Graphical Modeler*) with two steps: service area → extract buildings within distance. Save it; it is a clickable version of your script.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] 2- and 4-minute service areas and coverage %
- [ ] North/south gap analysis
- [ ] Best second-station site with numbers
- [ ] QGIS map + graphical model (.model3)

### 🪞 Reflection — your long-term plan

Which QGIS tools do you already use by clicking? Pick one of your own workflows and write the list of `native:` tool ids it uses — that is your first script.

✍️ **Your answer:** …